In [1]:
"""
================================================================================
 OPTIONS-IMPLIED PRICE PATH HEATMAP
 --------------------------------------------------------------------------
 Builds the market-implied (risk-neutral) distribution of a ticker's future
 price from its live option chain and overlays it as a heatmap on the
 historical price series.

 METHOD
   1. Pull the full option chain (all expiries) from Yahoo Finance.
   2. Recover the implied forward per expiry via put-call parity
      (F = K + e^{rT}(C - P)), which absorbs dividends and carry.
   3. Fit a smoothed total-variance smile w(k) = sigma(k)^2 * T on
      log-moneyness k = ln(K/F), using OTM options only.
   4. Interpolate the smile across expiries in total variance to obtain a
      continuous surface w(k, T) -> daily density slices.
   5. Apply Breeden-Litzenberger: q(K) = e^{rT} * d2C/dK2, giving the
      risk-neutral PDF of the terminal price at each horizon.
   6. Render the PDF slices as a heatmap cone with quantile paths.

 NOTE: this is the RISK-NEUTRAL density, not a real-world forecast. It is
 the market's pricing measure — it embeds the variance risk premium, so the
 left tail is fatter than any physical-measure estimate.
================================================================================
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from scipy.interpolate import UnivariateSpline
from scipy.optimize import brentq
from scipy.stats import norm

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. AESTHETIC CONFIGURATION
# ==============================================================================
bg_color = '#0b0f19'
grid_color = '#1f2937'
cyan_neon = '#00f2ea'
magenta_neon = '#ff007f'
yellow_neon = '#ffd700'
text_color = '#e5e7eb'

NEON_SCALE = [
    [0.00, bg_color],
    [0.12, '#0d1b3e'],
    [0.30, '#0b5f87'],
    [0.50, cyan_neon],
    [0.72, magenta_neon],
    [1.00, yellow_neon],
]


@dataclass
class Config:
    """Runtime parameters for the surface build."""
    ticker: str = "SPY"
    lookback: str = "6mo"          # history window for the price line
    min_dte: int = 5               # ignore near-worthless expiries
    max_dte: int = 400             # horizon cap in calendar days
    max_expiries: int = 14         # expiries sampled across the term structure
    risk_free: float | None = None  # None -> pull 13-week T-bill (^IRX)
    min_open_interest: int = 1
    max_spread_pct: float = 0.90   # reject quotes wider than this (rel. to mid)
    n_price: int = 340             # vertical resolution of the heatmap
    n_fine: int = 1201             # strike nodes used for the BL derivative
    band_sd: float = 3.6           # vertical extent, in terminal std devs
    col_normalize: bool = True     # scale each day to its own max density
    quantiles: tuple = (0.05, 0.25, 0.50, 0.75, 0.95)
    k_grid: np.ndarray = field(
        default_factory=lambda: np.linspace(-2.0, 2.0, 241))


# ==============================================================================
# 2. BLACK-76 CORE
# ==============================================================================
def b76_undisc(F, K, T, sigma, is_call=True):
    """Undiscounted Black-76 price. Vectorised over K and sigma."""
    F = np.asarray(F, dtype=float)
    K = np.asarray(K, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    sq = np.maximum(sigma, 1e-9) * np.sqrt(max(T, 1e-9))
    d1 = (np.log(F / K) + 0.5 * sq ** 2) / sq
    d2 = d1 - sq
    if is_call:
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    return K * norm.cdf(-d2) - F * norm.cdf(-d1)


def implied_vol(price_undisc, F, K, T, is_call=True):
    """Invert Black-76 on an undiscounted premium. Returns NaN on failure."""
    intrinsic = max(F - K, 0.0) if is_call else max(K - F, 0.0)
    upper = F if is_call else K
    if not (intrinsic + 1e-8 < price_undisc < upper - 1e-8):
        return np.nan
    try:
        return brentq(
            lambda s: b76_undisc(F, K, T, s, is_call) - price_undisc,
            1e-3, 6.0, xtol=1e-6, maxiter=100,
        )
    except (ValueError, RuntimeError):
        return np.nan


# ==============================================================================
# 3. DATA ACQUISITION
# ==============================================================================
def get_risk_free() -> float:
    """13-week T-bill yield as a continuous approximation."""
    try:
        irx = yf.Ticker("^IRX").history(period="5d")["Close"].dropna()
        if len(irx):
            return float(irx.iloc[-1]) / 100.0
    except Exception:
        pass
    return 0.043


def mid_price(df: pd.DataFrame) -> pd.Series:
    """Bid-ask mid where the quote is sane, else last traded price."""
    bid, ask, last = df["bid"], df["ask"], df["lastPrice"]
    mid = (bid + ask) / 2.0
    good = (bid > 0) & (ask > 0) & (ask >= bid)
    return mid.where(good, last)


def load_chains(tk: yf.Ticker, cfg: Config, spot: float, today: pd.Timestamp):
    """Return [(T_years, DataFrame), ...] of cleaned chains, sorted by T."""
    expiries = list(tk.options)
    if not expiries:
        raise RuntimeError(f"No listed options found for {cfg.ticker}.")

    valid = []
    for exp in expiries:
        dte = (pd.Timestamp(exp) - today).days
        if cfg.min_dte <= dte <= cfg.max_dte:
            valid.append((exp, dte))
    if not valid:
        raise RuntimeError("No expiries inside the requested DTE window.")

    # Sample evenly across the term structure rather than taking the front N.
    if len(valid) > cfg.max_expiries:
        idx = np.unique(np.linspace(0, len(valid) - 1, cfg.max_expiries).astype(int))
        valid = [valid[i] for i in idx]

    chains = []
    for exp, dte in valid:
        try:
            oc = tk.option_chain(exp)
        except Exception:
            continue
        calls, puts = oc.calls.copy(), oc.puts.copy()
        calls["cp"], puts["cp"] = "C", "P"
        df = pd.concat([calls, puts], ignore_index=True)

        for col in ("bid", "ask", "lastPrice", "openInterest", "volume"):
            if col not in df:
                df[col] = 0.0
        df[["bid", "ask", "lastPrice", "openInterest", "volume"]] = (
            df[["bid", "ask", "lastPrice", "openInterest", "volume"]].fillna(0.0))

        df["mid"] = mid_price(df)
        spread = (df["ask"] - df["bid"]) / df["mid"].replace(0, np.nan)

        keep = (
            (df["mid"] > 0.005)
            & (df["strike"] > 0.25 * spot)
            & (df["strike"] < 3.0 * spot)
            & ((df["openInterest"] >= cfg.min_open_interest) | (df["volume"] > 0))
            & (spread.fillna(9.9) < cfg.max_spread_pct)
        )
        df = df.loc[keep, ["strike", "cp", "mid", "openInterest", "volume"]]
        if len(df) >= 8:
            chains.append((dte / 365.0, df.reset_index(drop=True)))

    if len(chains) == 0:
        raise RuntimeError("Chains loaded but nothing survived the liquidity filter.")
    return sorted(chains, key=lambda x: x[0])


# ==============================================================================
# 4. FORWARD & SMILE CONSTRUCTION
# ==============================================================================
def implied_forward(df: pd.DataFrame, T: float, r: float, spot: float) -> float:
    """Forward from put-call parity on the strikes nearest the money."""
    piv = df.pivot_table(index="strike", columns="cp", values="mid").dropna()
    if piv.empty or not {"C", "P"}.issubset(piv.columns):
        return spot * np.exp(r * T)
    piv = piv.assign(dist=np.abs(piv.index - spot)).nsmallest(8, "dist")
    fwd = piv.index.values + (piv["C"] - piv["P"]).values * np.exp(r * T)
    F = float(np.median(fwd))
    # Sanity guard: parity can blow up on stale quotes.
    return F if 0.5 * spot < F < 2.0 * spot else spot * np.exp(r * T)


def fit_smile(df: pd.DataFrame, F: float, T: float, cfg: Config, r: float):
    """OTM-only total-variance smile, returned on cfg.k_grid."""
    otm = df[((df.cp == "C") & (df.strike >= F)) | ((df.cp == "P") & (df.strike < F))]
    disc = np.exp(-r * T)   # market premia are discounted; Black-76 core is not

    rows = []
    for K, cp, mid, oi, vol in otm[
            ["strike", "cp", "mid", "openInterest", "volume"]].itertuples(index=False):
        iv = implied_vol(mid / disc, F, K, T, is_call=(cp == "C"))
        if np.isfinite(iv) and 0.02 < iv < 4.0:
            rows.append((np.log(K / F), iv, max(oi, 0) + max(vol, 0) + 1.0))
    if len(rows) < 5:
        return None

    k, iv, wt = (np.array(x) for x in zip(*rows))
    order = np.argsort(k)
    k, iv, wt = k[order], iv[order], wt[order]

    # Collapse duplicate strikes (call/put overlap at the ATM boundary).
    k, inv = np.unique(k, return_inverse=True)
    iv = np.bincount(inv, weights=iv * wt) / np.bincount(inv, weights=wt)
    wt = np.bincount(inv, weights=wt)
    if len(k) < 5:
        return None

    w = iv ** 2 * T                       # total implied variance
    sw = np.sqrt(wt / wt.max())           # de-emphasise illiquid wings

    try:
        spl = UnivariateSpline(k, w, w=sw, k=3, s=len(k) * np.var(w) * 0.05)
        w_fit = spl(cfg.k_grid)
    except Exception:
        w_fit = np.interp(cfg.k_grid, k, w)

    # Flat-in-vol extrapolation outside the quoted strike range.
    lo, hi = k.min(), k.max()
    w_edge_lo = float(np.interp(lo, k, w))
    w_edge_hi = float(np.interp(hi, k, w))
    w_fit = np.where(cfg.k_grid < lo, w_edge_lo, w_fit)
    w_fit = np.where(cfg.k_grid > hi, w_edge_hi, w_fit)

    w_fit = np.maximum(w_fit, 1e-8)
    atm_iv = float(np.sqrt(np.interp(0.0, cfg.k_grid, w_fit) / T))
    return w_fit, atm_iv, len(k)


# ==============================================================================
# 5. IMPLIED SURFACE
# ==============================================================================
class ImpliedSurface:
    """w(k, T) interpolated linearly in total variance across expiries."""

    def __init__(self, times, forwards, w_matrix, k_grid, spot):
        self.T = np.asarray(times, dtype=float)
        self.F = np.asarray(forwards, dtype=float)
        self.W = np.asarray(w_matrix, dtype=float)      # (n_expiry, n_k)
        self.k = np.asarray(k_grid, dtype=float)
        self.spot = float(spot)
        # Implied carry rate per expiry, used for forward interpolation.
        self.carry = np.log(self.F / self.spot) / self.T

    def forward(self, T: float) -> float:
        c = float(np.interp(T, self.T, self.carry))
        return self.spot * np.exp(c * T)

    def total_var(self, k, T: float) -> np.ndarray:
        k = np.atleast_1d(np.asarray(k, dtype=float))
        if T <= self.T[0]:
            slice_w = self.W[0] * (T / self.T[0])
        elif T >= self.T[-1]:
            slice_w = self.W[-1] * (T / self.T[-1])
        else:
            j = int(np.searchsorted(self.T, T)) - 1
            a = (T - self.T[j]) / (self.T[j + 1] - self.T[j])
            slice_w = (1 - a) * self.W[j] + a * self.W[j + 1]
        return np.maximum(np.interp(k, self.k, slice_w), 1e-10)

    def atm_vol(self, T: float) -> float:
        return float(np.sqrt(self.total_var(0.0, T)[0] / max(T, 1e-9)))


def build_surface(cfg: Config, tk: yf.Ticker, spot: float, r: float,
                  today: pd.Timestamp):
    chains = load_chains(tk, cfg, spot, today)
    times, fwds, wmat, diag = [], [], [], []

    for T, df in chains:
        F = implied_forward(df, T, r, spot)
        fit = fit_smile(df, F, T, cfg, r)
        if fit is None:
            continue
        w_fit, atm_iv, n_used = fit
        times.append(T)
        fwds.append(F)
        wmat.append(w_fit)
        diag.append({"dte": int(round(T * 365)), "F": F, "atm_iv": atm_iv,
                     "n_strikes": n_used})

    if len(times) < 2:
        raise RuntimeError("Fewer than two usable expiries — cannot interpolate.")
    return ImpliedSurface(times, fwds, wmat, cfg.k_grid, spot), pd.DataFrame(diag)


# ==============================================================================
# 6. BREEDEN-LITZENBERGER DENSITY
# ==============================================================================
def rn_density(surf: ImpliedSurface, T: float, cfg: Config):
    """Risk-neutral PDF at horizon T on an adaptive strike grid."""
    sd = max(np.sqrt(surf.total_var(0.0, T)[0]), 1e-4)
    k = np.linspace(-6.0 * sd, 6.0 * sd, cfg.n_fine)
    F = surf.forward(T)
    K = F * np.exp(k)

    iv = np.sqrt(surf.total_var(k, T) / max(T, 1e-9))
    C = b76_undisc(F, K, T, iv, is_call=True)

    q = np.gradient(np.gradient(C, K), K)   # d2C/dK2 -> undiscounted density
    q = np.clip(q, 0.0, None)
    area = np.trapezoid(q, K) if hasattr(np, "trapezoid") else np.trapz(q, K)
    if area <= 0:
        return K, np.zeros_like(K)
    return K, q / area


def density_quantiles(K, q, levels):
    dK = np.diff(K)
    cdf = np.concatenate([[0.0], np.cumsum(0.5 * (q[1:] + q[:-1]) * dK)])
    if cdf[-1] <= 0:
        return {lv: np.nan for lv in levels}
    cdf /= cdf[-1]
    return {lv: float(np.interp(lv, cdf, K)) for lv in levels}


def build_cone(surf: ImpliedSurface, cfg: Config, today: pd.Timestamp):
    """Daily density slices -> (dates, price_grid, Z, quantile frame)."""
    horizon = surf.T[-1]
    dates = pd.bdate_range(today + pd.Timedelta(days=1),
                           today + pd.Timedelta(days=int(horizon * 365)))
    if len(dates) == 0:
        raise RuntimeError("Empty forward date grid.")

    F_end, sd_end = surf.forward(horizon), np.sqrt(surf.total_var(0.0, horizon)[0])
    y = np.linspace(F_end * np.exp(-cfg.band_sd * sd_end),
                    F_end * np.exp(+cfg.band_sd * sd_end), cfg.n_price)

    Z = np.zeros((cfg.n_price, len(dates)))
    qrows = []
    for j, d in enumerate(dates):
        T = max((d - today).days / 365.0, 1.0 / 365.0)
        K, q = rn_density(surf, T, cfg)
        Z[:, j] = np.interp(y, K, q, left=0.0, right=0.0)
        qrows.append({"date": d, **{f"q{int(lv*100):02d}": v
                                    for lv, v in density_quantiles(
                                        K, q, cfg.quantiles).items()}})

    if cfg.col_normalize:
        peak = Z.max(axis=0, keepdims=True)
        Z = np.divide(Z, peak, out=np.zeros_like(Z), where=peak > 0)

    return dates, y, Z, pd.DataFrame(qrows).set_index("date")


# ==============================================================================
# 7. VISUALISATION
# ==============================================================================
def build_figure(cfg, hist, surf, dates, y, Z, qdf, spot, r):
    fig = go.Figure()

    fig.add_trace(go.Heatmap(
        x=dates, y=y, z=Z,
        colorscale=NEON_SCALE, zsmooth="best",
        name="Implied density",
        colorbar=dict(
            title=dict(text="RND<br>density", font=dict(color=text_color, size=11)),
            tickfont=dict(color=text_color, size=10),
            outlinewidth=0, thickness=14, len=0.62, y=0.5),
        hovertemplate="%{x|%d %b %Y}<br>Price %{y:,.2f}<br>Density %{z:.3f}<extra></extra>",
    ))

    # --- historical price ---
    fig.add_trace(go.Scatter(
        x=hist.index, y=hist["Close"], mode="lines", name=f"{cfg.ticker} close",
        line=dict(color=cyan_neon, width=2),
        hovertemplate="%{x|%d %b %Y}<br>%{y:,.2f}<extra></extra>"))

    # --- quantile paths ---
    style = {
        "q05": (magenta_neon, "dot", 1.6, "5%"),
        "q25": (yellow_neon, "dash", 1.4, "25%"),
        "q50": (text_color, "solid", 2.0, "Median"),
        "q75": (yellow_neon, "dash", 1.4, "75%"),
        "q95": (magenta_neon, "dot", 1.6, "95%"),
    }
    for col, (color, dash, wdt, label) in style.items():
        if col not in qdf:
            continue
        fig.add_trace(go.Scatter(
            x=qdf.index, y=qdf[col], mode="lines", name=label,
            line=dict(color=color, width=wdt, dash=dash),
            hovertemplate=f"{label}: %{{y:,.2f}}<extra></extra>"))

    # --- spot reference & anchor ---
    fig.add_hline(y=spot, line=dict(color=text_color, width=1, dash="dot"),
                  opacity=0.45)
    fig.add_vline(x=hist.index[-1], line=dict(color=text_color, width=1),
                  opacity=0.35)

    # --- listed expiry markers ---
    for T in surf.T:
        d = hist.index[-1] + pd.Timedelta(days=int(T * 365))
        fig.add_vline(x=d, line=dict(color=grid_color, width=1), opacity=0.55)

    fig.update_layout(
        template="plotly_dark",
        title=dict(
            text=(f"<b>{cfg.ticker}</b> — options-implied price distribution"
                  f"<br><sub>Breeden-Litzenberger risk-neutral density · "
                  f"spot {spot:,.2f} · ATM IV 30d "
                  f"{surf.atm_vol(30/365)*100:.1f}% · r {r*100:.2f}%</sub>"),
            font=dict(color=text_color, size=18)),
        paper_bgcolor=bg_color, plot_bgcolor=bg_color,
        font=dict(family="monospace", color=text_color, size=12),
        xaxis=dict(gridcolor=grid_color, zeroline=False, showspikes=True,
                   spikecolor=grid_color, spikethickness=1,
                   rangeslider=dict(visible=False)),
        yaxis=dict(gridcolor=grid_color, zeroline=False, title="Price",
                   range=[min(y.min(), hist["Close"].min() * 0.97),
                          max(y.max(), hist["Close"].max() * 1.03)]),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0,
                    bgcolor="rgba(0,0,0,0)"),
        hovermode="x unified", margin=dict(l=70, r=20, t=110, b=50), height=760,
    )
    return fig


# ==============================================================================
# 8. ORCHESTRATION
# ==============================================================================
def implied_path_heatmap(ticker="SPY", show=True, **kwargs):
    """Build the figure for `ticker`. Extra kwargs override Config fields."""
    cfg = Config(ticker=ticker.upper(), **kwargs)
    tk = yf.Ticker(cfg.ticker)

    hist = tk.history(period=cfg.lookback, auto_adjust=False)
    if hist.empty:
        raise RuntimeError(f"No price history returned for {cfg.ticker}.")
    hist.index = pd.to_datetime(hist.index).tz_localize(None)
    spot = float(hist["Close"].iloc[-1])
    today = hist.index[-1].normalize()

    r = cfg.risk_free if cfg.risk_free is not None else get_risk_free()
    surf, diag = build_surface(cfg, tk, spot, r, today)
    dates, y, Z, qdf = build_cone(surf, cfg, today)
    fig = build_figure(cfg, hist, surf, dates, y, Z, qdf, spot, r)

    # ---------------------------- TERMINAL SUMMARY ----------------------------
    last = qdf.iloc[-1]
    horizon_days = (qdf.index[-1] - today).days
    print("=" * 78)
    print(f" IMPLIED PATH SURFACE — {cfg.ticker}")
    print("=" * 78)
    print(f" Spot            : {spot:>12,.2f}   as of {today:%Y-%m-%d}")
    print(f" Risk-free (r)   : {r*100:>12.2f}%")
    print(f" Expiries used   : {len(surf.T):>12d}   "
          f"({diag.dte.min()}–{diag.dte.max()} DTE)")
    print(f" ATM IV  30d/90d : {surf.atm_vol(30/365)*100:>11.2f}% / "
          f"{surf.atm_vol(90/365)*100:.2f}%")
    print("-" * 78)
    print(" TERM STRUCTURE")
    print(f"   {'DTE':>5} {'Forward':>12} {'ATM IV':>9} {'Strikes':>9}")
    for _, row in diag.iterrows():
        print(f"   {row.dte:>5.0f} {row.F:>12,.2f} {row.atm_iv*100:>8.2f}% "
              f"{row.n_strikes:>9.0f}")
    print("-" * 78)
    print(f" RISK-NEUTRAL QUANTILES @ +{horizon_days}d ({qdf.index[-1]:%Y-%m-%d})")
    for col in ("q05", "q25", "q50", "q75", "q95"):
        if col in last:
            print(f"   {col[1:]:>3}%  {last[col]:>12,.2f}   "
                  f"({last[col]/spot - 1:+7.2%})")
    print("=" * 78)

    if show:
        fig.show()
    return fig, {"surface": surf, "term": diag, "quantiles": qdf,
                 "grid": (dates, y, Z)}


if __name__ == "__main__":
    fig, out = implied_path_heatmap("SPY", lookback="6mo", max_dte=270)
    fig.write_html("implied_path_heatmap.html", include_plotlyjs="cdn")

 IMPLIED PATH SURFACE — SPY
 Spot            :       776.34   as of 2026-08-14
 Risk-free (r)   :         3.70%
 Expiries used   :           14   (5–229 DTE)
 ATM IV  30d/90d :       13.69% / 14.34%
------------------------------------------------------------------------------
 TERM STRUCTURE
     DTE      Forward    ATM IV   Strikes
       5       776.10     7.44%       104
       6       776.15     7.94%        99
       7       776.36     9.36%       176
      17       776.98    10.77%       225
      21       777.56    11.93%       232
      35       777.69    14.23%       304
      42       777.60    12.61%       131
      63       779.18    14.16%       219
      77       780.33    14.15%       243
     108       782.77    14.53%       180
     126       783.85    15.83%       157
     154       785.62    16.07%       188
     168       786.84    15.54%       145
     229       791.04    16.58%       194
----------------------------------------------------------------------------